# PandaPickCube inference

This notebook shows how a Brax PPO policy improves while training on the `PandaPickCube` task: a Franka arm learns to reach, grasp, and lift a cube in MuJoCo.


## Verify imports and the render backend

### MuJoCo Playground + JAX/MJX
The models for the manipulation environment we load was trained on JAX. The notebook sets `impl=jax` so inference stays on the same path we used before for checkpoint compatibality. Exporting to another format like ONNX can be done if cross-simulator/framework interoperability is needed.

### Headless rendering
If your computer runs without a display, like this one, we can probe Mesa/OSMesa or EGL in a subprocess, and then render each rollout video outside the notebook kernel. The kernel itself sets `MUJOCO_GL=disable` so stepping stays lightweight.

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT))

from headless_gl import (
    build_gl_env,
    enable_demo_quiet_mode,
    probe_render_subprocess,
    quiet_demo_output,
)

enable_demo_quiet_mode()

CAN_RENDER, probe_report, GL_ENV = probe_render_subprocess(REPO_ROOT)
RENDER_BACKEND = probe_report if CAN_RENDER else None

if CAN_RENDER:
    print(f"Render backend: {RENDER_BACKEND}")
else:
    GL_ENV = build_gl_env(REPO_ROOT)
    print("No render backend worked, so section 4 will fail. Backends tried:\n")
    print(probe_report)

# This kernel only steps the environment; section 4 renders in subprocesses.
os.environ["MUJOCO_GL"] = "disable"

import imageio_ffmpeg

ffmpeg_dir = str(Path(imageio_ffmpeg.get_ffmpeg_exe()).parent)
os.environ["PATH"] = ffmpeg_dir + os.pathsep + os.environ.get("PATH", "")

with quiet_demo_output():
    import jax
    import mujoco
    from mujoco_playground import registry

import numpy as np
from IPython.display import Video, display

print("Working directory:", REPO_ROOT)
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
for package in ("mujoco", "mujoco-mjx", "jax", "jaxlib", "brax", "playground"):
    print(f"{package}: {version(package)}")


## 1. Confirm checkpoint paths

Brax saves PPO policies as Orbax checkpoint folders. Each folder contains `ppo_network_config.json` plus weight shards.

The next bit of Python defines the three policies we will compare in section 4: early, mid, and final, and verifies they are present on disk.


In [ ]:
CHECKPOINT_STAGES = [
    {
        "label": "early — weak policy",
        "path": REPO_ROOT / "PandaPickCube-20260807-131132/checkpoints/000008192000",
        "video": "panda_pick_cube_early.mp4",
    },
    {
        "label": "mid — improving policy",
        "path": REPO_ROOT / "PandaPickCube-20260817-150103/checkpoints/000006553600",
        "video": "panda_pick_cube_mid.mp4",
    },
    {
        "label": "final — strong policy",
        "path": REPO_ROOT / "PandaPickCube-20260817-150103/checkpoints/000045875200",
        "video": "panda_pick_cube.mp4",
    },
]

for index, stage in enumerate(CHECKPOINT_STAGES, start=1):
    checkpoint = stage["path"]
    if not (checkpoint / "ppo_network_config.json").is_file():
        raise FileNotFoundError(
            f"Stage {index} missing ppo_network_config.json: {checkpoint}\n"
            "Rebuild the course Docker image so both checkpoint archives are extracted."
        )
    print(f"{index}. {stage['label']}")
    print(f"   checkpoint: {checkpoint}")
    print(f"   video:      {stage['video']}")


## 2. Load `PandaPickCube`

`PandaPickCube` comes from [MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground): a tabletop scene with a Franka Panda arm, a free cube, and a sparse reward for moving the cube toward a target pose.

**Menagerie assets**: the Franka model lives in [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie), the digital twin of the Franka Emika Panda robot.

Notice the observation size, action size, timestep, and episode length. That frames what the policy network sees and how long each rollout can run (150 steps × 0.02 s ≈ 3 s of sim time).


In [ ]:
ENV_NAME = "PandaPickCube"
env_cfg = registry.get_default_config(ENV_NAME)
with quiet_demo_output():
    env = registry.load(
        ENV_NAME,
        config=env_cfg,
        config_overrides={"impl": "jax"},
    )

print(
    f"Loaded {ENV_NAME}: impl={env._config.impl}, "
    f"obs={env.observation_size}, actions={env.action_size}, "
    f"dt={env.dt}s, episode_length={int(env_cfg.episode_length)}"
)
print("MuJoCo version:", mujoco.__version__)


## 3. PPO policy loader

Training saved three things we need at inference time:

1. **Network architecture**: in `ppo_network_config.json` (layer sizes, activation, PPO hyperparameters metadata).
2. **Learned weights**: Orbax checkpoint shards in the same folder.
3. **Environment contract**: observation and action sizes must match the live `PandaPickCube` env.

The helper in the next cell rebuilds the Brax actor, loads weights, and returns a deterministic policy function.

We load a **fresh** policy for each checkpoint in section 4.


In [ ]:
import json

from brax.training import checkpoint as brax_checkpoint
from brax.training.agents.ppo import networks as ppo_networks
from ml_collections import config_dict


def load_ppo_policy(checkpoint_path, env, deterministic=True):
    """Load Brax PPO policy; tolerate malformed Brax 0.14.2 checkpoint JSON."""
    path = Path(checkpoint_path)
    loaded_dict = json.loads((path / "ppo_network_config.json").read_text())
    factory_kwargs = loaded_dict["network_factory_kwargs"]

    if "activation" in factory_kwargs:
        factory_kwargs["activation"] = brax_checkpoint.networks.ACTIVATION[
            factory_kwargs["activation"]
        ]

    for init_fn_name in brax_checkpoint._KERNEL_INIT_FN_KEYWORDS:
        if init_fn_name not in factory_kwargs:
            continue
        init_fn_value = factory_kwargs[init_fn_name]
        if init_fn_value is None:
            del factory_kwargs[init_fn_name]
            continue
        factory_kwargs[init_fn_name] = brax_checkpoint.networks.KERNEL_INITIALIZER[
            init_fn_value
        ]

    loaded_dict["observation_size"] = env.observation_size
    loaded_dict["action_size"] = env.action_size

    config = config_dict.create(**loaded_dict)
    params = brax_checkpoint.load(path)
    ppo_network = brax_checkpoint.get_network(config, ppo_networks.make_ppo_networks)
    make_inference_fn = ppo_networks.make_inference_fn(ppo_network)
    return make_inference_fn(params, deterministic=deterministic)


print("PPO loader ready.")


## 4. Roll out and render each checkpoint

Our three checkpoints:

| Stage | What to say while the video plays |
|-------|-----------------------------------|
| Early | Unreliable grasp |
| Mid | Clear improvement, better approach and contact |
| Final | Stable pick-and-place; this is the policy you would ship. |


The next cell defines rollout helpers. Then run **4a → 4b → 4c** one at a time.  
Each code cell loads a checkpoint, rolls out one episode (`SEED=42`), and outputs its video.  

Here, videos are generated from trajectory `.npz` files let us re-render without rerunning inference.

The first JIT compile per checkpoint can take up to a minute.


In [ ]:
import subprocess
import sys

NUM_EPISODES = 1
SEED = 42

jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
progression_results: list[dict] = []

if not CAN_RENDER:
    raise RuntimeError(
        "No working render backend. Rebuild the course Docker image or "
        "restart the kernel, then rerun section 1."
    )


def rollout_episode(jit_policy, seed: int) -> tuple[float, list]:
    rng = jax.random.PRNGKey(seed)
    episodes = []
    for ep in range(NUM_EPISODES):
        rng, reset_rng = jax.random.split(rng)
        state = jit_reset(reset_rng)
        trajectory = [state]
        episode_reward = 0.0
        for _ in range(int(env_cfg.episode_length)):
            rng, action_rng = jax.random.split(rng)
            action, _ = jit_policy(state.obs, action_rng)
            state = jit_step(state, action)
            trajectory.append(state)
            episode_reward += float(np.asarray(state.reward))
            if bool(np.asarray(state.done)):
                break
        episodes.append((episode_reward, trajectory))
        print(f"  Episode {ep + 1}: reward={episode_reward:.3f}  steps={len(trajectory) - 1}")
    return max(episodes, key=lambda item: item[0])


def save_trajectory_npz(path: Path, trajectory, episode_reward: float) -> None:
    qpos = np.stack([np.asarray(s.data.qpos) for s in trajectory])
    qvel = np.stack([np.asarray(s.data.qvel) for s in trajectory])
    mocap_pos = np.stack([np.asarray(s.data.mocap_pos) for s in trajectory])
    mocap_quat = np.stack([np.asarray(s.data.mocap_quat) for s in trajectory])
    rewards = np.array(
        [float(np.asarray(s.reward)) for s in trajectory[1:]], dtype=np.float32
    )
    np.savez(
        path,
        qpos=qpos,
        qvel=qvel,
        mocap_pos=mocap_pos,
        mocap_quat=mocap_quat,
        rewards=rewards,
        episode_reward=episode_reward,
        dt=float(env.dt),
        env_name=ENV_NAME,
    )
    print(f"  Saved trajectory to {path} ({qpos.shape[0]} steps)")


def render_trajectory(traj_path: Path, video_path: Path) -> None:
    result = subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / "scripts" / "render_trajectory.py"),
            str(traj_path),
            "-o",
            str(video_path),
            "--env",
            ENV_NAME,
        ],
        cwd=REPO_ROOT,
        env=GL_ENV,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout.strip())
    if result.returncode != 0:
        raise RuntimeError(
            f"Rendering failed with backend {RENDER_BACKEND!r}.\n\n"
            f"{result.stderr or result.stdout}"
        )


def run_checkpoint_stage(stage: dict) -> dict:
    """Load one checkpoint, roll out, render, and show the video inline."""
    print(f"\n=== {stage['label']} ===")
    checkpoint = stage["path"]
    video_path = REPO_ROOT / stage["video"]
    traj_path = video_path.with_suffix(".npz")

    inference_fn = load_ppo_policy(checkpoint, env, deterministic=True)
    jit_policy = jax.jit(inference_fn)

    episode_reward, trajectory = rollout_episode(jit_policy, SEED)
    save_trajectory_npz(traj_path, trajectory, episode_reward)
    render_trajectory(traj_path, video_path)

    display(Video(str(video_path), embed=True, width=640))
    print(f"  Done: {video_path}")

    result = {
        "label": stage["label"],
        "reward": episode_reward,
        "steps": len(trajectory) - 1,
        "video": video_path,
    }
    progression_results.append(result)
    return result


print("Rollout helpers ready. Run the next three cells when you are ready for each video.")


### 4a. Early checkpoint, weak policy

In [ ]:
run_checkpoint_stage(CHECKPOINT_STAGES[0])

### 4b. Mid checkpoint, improving policy

In [ ]:
run_checkpoint_stage(CHECKPOINT_STAGES[1])

### 4c. Final checkpoint, strong policy

In [ ]:
run_checkpoint_stage(CHECKPOINT_STAGES[2])

In [ ]:
print("=== Checkpoint progression summary ===")
for row in progression_results:
    print(f"{row['label']}: reward={row['reward']:.3f}, steps={row['steps']}, video={row['video'].name}")

**Remarks**

- Same environment, same seed, different checkpoints → behavior change is purely from learning.